# Booster Impact Tracker — Velocity Frame Edition (v2.0)

Run each cell with **Shift+Enter**.  
Change any parameter and re-run from that cell down (`Run → Run All Below`).

### Velocity-frame recap
| Symbol | Name | Description |
|--------|------|-------------|
| **γV** | Flight-path angle | Angle of velocity vector above horizontal (deg). Positive = climbing. |
| **γH** | Heading azimuth | Direction of horizontal velocity (deg). 0 = North, 90 = East. |
| **ê_t** | Tangential axis | Along velocity vector |
| **ê_nV** | Pitch-normal axis | ⊥ to velocity, upward in pitch plane — lift & pitch commands act here |
| **ê_nH** | Yaw-normal axis | Lateral direction — yaw commands act here |

## 1 — Imports

In [ ]:
from sim.simulate import simulate
from sim.physics import Params
from plot import plot_results
from dataclasses import replace
import matplotlib.pyplot as plt
%matplotlib inline
print('Imports OK')

## 2 — Parameters

> **Gravity turn tip:** use `launch_angle` ≥ 85° with `grav_turn=True`.  
> At 75° the rapid pitch-over at low speed keeps max altitude very low.  
> With `grav_turn=False` any angle works (thrust held at fixed body angle).

In [ ]:
params = Params(
    # ── Booster ──────────────────────────────────────
    m_pay        = 300,    # kg  payload mass
    m_prop       = 5_000,  # kg  propellant mass
    m_str        = 800,    # kg  structural dry mass
    # ── Motor ────────────────────────────────────────
    isp          = 260,    # s
    thrust_kn    = 120,    # kN
    burn_max     = 60,     # s
    # ── Aerodynamics ─────────────────────────────────
    cd           = 0.4,    # drag coeff – powered
    cd_fall      = 1.2,    # drag coeff – tumbling post-burnout
    cl           = 0.0,    # lift coefficient  (0 = no lift)
    cd_ctrl      = 0.3,    # actuator drag (ZEM guidance)
    diam         = 1.2,    # m  reference diameter
    atm          = 'isa',  # 'isa' | 'exp' | 'none'
    # ── Launch angles ─────────────────────────────────
    launch_angle   = 85,   # deg  initial γV (elevation)
    launch_azimuth = 90,   # deg  initial γH (0=N, 90=E)
    # ── Velocity-frame dynamics ───────────────────────
    grav_turn       = True,  # True → thrust ∥ velocity; False → fixed body angle
    grav_turn_v_min = 30.0,  # m/s  minimum speed before gravity turn activates
    a_cmd_t         = 0.0,   # m/s²  tangential accel command (ê_t)
    a_cmd_nv        = 0.0,   # m/s²  pitch-normal accel command (ê_nV)
    a_cmd_nh        = 0.0,   # m/s²  yaw-normal accel command (ê_nH)
    # ── ZEM guidance target ───────────────────────────
    x_target     = 0,      # km  downrange
    y_target     = 60,     # km  crossrange
    z_target     = 0,      # km  altitude
    a_lat_max    = 2.0,    # g   ZEM lateral accel limit
)

## 3 — Run simulation

In [ ]:
result = simulate(params)

print('=== Summary ===')
for k, v in result['summary'].items():
    print(f'  {k:<30} {v}')

## 4 — Plots (7-panel dashboard)

In [ ]:
plot_results(result, params)

## 5 — Inspect time-series
All arrays live in `result['series']`.

In [ ]:
s = result['series']
print('Available keys:', list(s.keys()))
print(f"Samples: {len(s['t'])}  |  Flight time: {s['t'][-1]} s")

In [ ]:
i = 10   # <-- change to inspect any timestep

print(f"t = {s['t'][i]} s")
print(f"  Position :  h={s['h'][i]} km   x={s['x'][i]} km   y={s['y'][i]} km")
print(f"  Velocity :  v={s['v'][i]} m/s")
print(f"  Frame    :  γV={s['gamma_v'][i]}°   γH={s['gamma_h'][i]}°")
print(f"  Forces   :  thrust={s['thr'][i]} kN   drag={s['drag_aero'][i]} kN   lift={s['lift'][i]} kN")
print(f"  ZEM      :  |aLat|={s['aLat'][i]} g   (x={s['aLat_x'][i]} g, y={s['aLat_y'][i]} g, z={s['aLat_z'][i]} g)")
print(f"  Accel    :  {s['acc'][i]} g")

## 6 — Flight-path angle profile (γV and γH)
Shows how the velocity-frame angles evolve over the full flight.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(s['t'], s['gamma_v'], color='#378ADD', linewidth=2, label='γV — flight-path angle')
ax.plot(s['t'], s['gamma_h'], color='#7F77DD', linewidth=1.5, linestyle='--', label='γH — heading azimuth')
ax.axhline(0, color='#999', linewidth=0.8, linestyle=':')
ax.set_xlabel('Time (s)')
ax.set_ylabel('degrees')
ax.set_title('Velocity-frame angles vs time')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7 — Gravity turn vs fixed-angle comparison
Runs the same vehicle twice and overlays the altitude and γV profiles.

In [ ]:
r_gt  = simulate(replace(params, grav_turn=True,  a_lat_max=0))  # no ZEM, pure ballistic
r_fix = simulate(replace(params, grav_turn=False, a_lat_max=0))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Altitude
ax1.plot(r_gt['series']['t'],  r_gt['series']['h'],  color='#378ADD', label='Gravity turn')
ax1.plot(r_fix['series']['t'], r_fix['series']['h'], color='#D85A30', linestyle='--', label='Fixed angle')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Altitude (km)')
ax1.set_title('Altitude comparison'); ax1.legend(); ax1.grid(alpha=0.3)

# γV
ax2.plot(r_gt['series']['t'],  r_gt['series']['gamma_v'],  color='#378ADD', label='Gravity turn')
ax2.plot(r_fix['series']['t'], r_fix['series']['gamma_v'], color='#D85A30', linestyle='--', label='Fixed angle')
ax2.axhline(0, color='#999', linewidth=0.8, linestyle=':')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('γV (deg)')
ax2.set_title('Flight-path angle γV'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

for label, r in [('Gravity turn', r_gt), ('Fixed angle', r_fix)]:
    sm = r['summary']
    print(f"{label:<15}  max_alt={sm['max_altitude_km']:.1f} km  "
          f"burnout_gV={sm['burnout_gamma_v_deg']:.1f}°  "
          f"range={sm['impact_range_km']:.1f} km")

## 8 — Lift coefficient sweep
Shows how increasing CL slows the gravity-turn pitch-over (higher γV at burnout → more altitude).

In [ ]:
cl_values = [0.0, 0.2, 0.5, 1.0]
colors     = ['#378ADD', '#1D9E75', '#D85A30', '#7F77DD']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

print(f"{'CL':>5}  {'max_alt':>10}  {'burnout_gV':>12}  {'max_lift':>10}  {'range':>10}")
for cl, col in zip(cl_values, colors):
    r  = simulate(replace(params, cl=cl, a_lat_max=0))
    sm = r['summary']
    sv = r['series']
    ax1.plot(sv['t'], sv['h'],       color=col, label=f'CL={cl}')
    ax2.plot(sv['t'], sv['gamma_v'], color=col, label=f'CL={cl}')
    print(f"{cl:>5}  {sm['max_altitude_km']:>10.2f}  "
          f"{sm['burnout_gamma_v_deg']:>12.1f}°  "
          f"{sm['max_lift_kn']:>10.3f} kN  "
          f"{sm['impact_range_km']:>10.1f} km")

ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Altitude (km)')
ax1.set_title('Altitude vs CL'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.axhline(0, color='#999', linewidth=0.8, linestyle=':')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('γV (deg)')
ax2.set_title('Flight-path angle γV vs CL'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9 — Pitch acceleration command sweep (a_cmd_nv)
`a_cmd_nv` acts along **ê_nV** (pitch-normal). Positive pulls the flight path upward, slowing the gravity-turn pitch-over.

In [ ]:
nv_values = [-2.0, 0.0, 2.0, 5.0]
colors     = ['#D85A30', '#378ADD', '#1D9E75', '#7F77DD']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

print(f"{'a_nV':>6}  {'max_alt':>10}  {'burnout_gV':>12}  {'range':>10}")
for nv, col in zip(nv_values, colors):
    r  = simulate(replace(params, a_cmd_nv=nv, a_lat_max=0))
    sm = r['summary']
    sv = r['series']
    ax1.plot(sv['t'], sv['h'],       color=col, label=f'a_nV={nv:+.1f} m/s²')
    ax2.plot(sv['t'], sv['gamma_v'], color=col, label=f'a_nV={nv:+.1f} m/s²')
    print(f"{nv:>+6.1f}  {sm['max_altitude_km']:>10.2f}  "
          f"{sm['burnout_gamma_v_deg']:>12.1f}°  "
          f"{sm['impact_range_km']:>10.1f} km")

ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Altitude (km)')
ax1.set_title('Altitude vs a_cmd_nV'); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax2.axhline(0, color='#999', linewidth=0.8, linestyle=':')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('γV (deg)')
ax2.set_title('Flight-path angle γV vs a_cmd_nV'); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 10 — Yaw command sweep (a_cmd_nh)
`a_cmd_nh` acts along **ê_nH** (yaw-normal). It changes γH over time, diverting the booster laterally. The impact range uses 3-D distance √(x²+y²).

In [ ]:
import math
nh_values = [0.0, 1.0, 3.0, -2.0]
colors     = ['#378ADD', '#1D9E75', '#D85A30', '#7F77DD']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

print(f"{'a_nH':>6}  {'final_gH':>10}  {'impact_x':>10}  {'impact_y':>10}  {'range':>10}")
for nh, col in zip(nh_values, colors):
    r  = simulate(replace(params, a_cmd_nh=nh, a_lat_max=0))
    sm = r['summary']
    sv = r['series']
    # Ground track
    ax1.plot(sv['x'], sv['y'], color=col, label=f'a_nH={nh:+.1f} m/s²')
    ax2.plot(sv['t'], sv['gamma_h'], color=col, label=f'a_nH={nh:+.1f} m/s²')
    print(f"{nh:>+6.1f}  {sm['final_gamma_h_deg']:>10.1f}°  "
          f"{sm['impact_x_km']:>10.2f} km  "
          f"{sm['impact_y_km']:>10.2f} km  "
          f"{sm['impact_range_km']:>10.1f} km")

ax1.set_xlabel('North X (km)'); ax1.set_ylabel('East Y (km)')
ax1.set_title('Ground track vs a_cmd_nH'); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('γH (deg)')
ax2.set_title('Heading azimuth γH vs a_cmd_nH'); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 11 — Launch angle sweep
Classic trade: higher angle → more altitude, less range.

In [ ]:
angles = [75, 80, 85, 88]

print(f"{'angle':>6}  {'max_alt':>10}  {'burnout_gV':>12}  {'burnout_alt':>12}  {'range':>10}  {'t_flight':>10}")
for angle in angles:
    r  = simulate(replace(params, launch_angle=angle, a_lat_max=0))
    sm = r['summary']
    print(f"{angle:>6}°  {sm['max_altitude_km']:>10.2f}  "
          f"{sm['burnout_gamma_v_deg']:>12.1f}°  "
          f"{sm['burnout_altitude_km']:>12.2f} km  "
          f"{sm['impact_range_km']:>10.1f} km  "
          f"{sm['flight_time_s']:>10.1f} s")

## 12 — Run check vectors (pytest)

In [ ]:
!python -m pytest tests/check_vectors.py -v